# ChatGPT Review Analysis — Reproducible Python Project

**Objective:** analyse customer reviews using data cleaning, exploratory analysis, TextBlob sentiment polarity and subjectivity, keyword extraction, rating comparisons, and monthly trends.

> Every figure used in the presentation should be generated by the cells in this notebook. Do not type results into the presentation until you have run the notebook and verified the outputs.

**Dataset:** `chatgpt_reviews.csv` (columns in the supplied file: Review Id, Review, Ratings, Review Date).

**Method note:** TextBlob is a lexicon/rule-based baseline. Its polarity labels are not ground-truth customer satisfaction labels. We will compare them with star ratings and inspect limitations.

## 0. Install and import libraries

Run this cell once in Google Colab. If you are using a local Jupyter environment, install the same packages in your environment.

In [ ]:
%pip -q install textblob scikit-learn wordcloud

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully")

## 1. Load the CSV

In Google Colab, upload the supplied CSV when prompted. If the file is already in the notebook's working directory, set `CSV_PATH` directly.

In [ ]:
# Google Colab upload option
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
except ImportError:
    # Local Jupyter option: put chatgpt_reviews.csv beside this notebook
    CSV_PATH = "chatgpt_reviews.csv"

raw_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns")
display(raw_df.head())

## 2. Understand the dataset structure

This step answers: How many rows and columns do we have? What are the fields, data types, date range, and rating values?

In [ ]:
print("Shape:", raw_df.shape)
print("\nColumns:", raw_df.columns.tolist())
print("\nData types:")
print(raw_df.dtypes)
print("\nInfo:")
raw_df.info()
print("\nRating values:", sorted(raw_df["Ratings"].dropna().unique().tolist()))
print("\nDate range (raw parse):")
raw_dates = pd.to_datetime(raw_df["Review Date"], errors="coerce")
print(raw_dates.min(), "to", raw_dates.max())

## 3. Data-quality audit

We will distinguish **exact duplicate rows** from **repeated Review IDs**. A repeated ID is not automatically safe to delete: the review text or date may differ. We will report both, remove only exact duplicate rows from the main analysis, and retain an audit copy of the raw data.

In [ ]:
audit = pd.Series({
    "raw_rows": len(raw_df),
    "raw_columns": raw_df.shape[1],
    "missing_review_id": raw_df["Review Id"].isna().sum(),
    "missing_review_text": raw_df["Review"].isna().sum(),
    "missing_rating": raw_df["Ratings"].isna().sum(),
    "missing_review_date": raw_df["Review Date"].isna().sum(),
    "exact_duplicate_rows": raw_df.duplicated().sum(),
    "repeated_review_id_rows_after_first": raw_df["Review Id"].duplicated().sum(),
    "distinct_review_ids_with_repeats": raw_df.loc[raw_df["Review Id"].duplicated(keep=False), "Review Id"].nunique(),
})
display(audit.to_frame("count"))

print("Rating range:", raw_df["Ratings"].min(), "to", raw_df["Ratings"].max())
print("Ratings outside 1–5:", (~pd.to_numeric(raw_df["Ratings"], errors="coerce").between(1, 5)).sum())

## 4. Prepare an analysis dataset

Decisions for this baseline analysis:
1. Keep `raw_df` unchanged for traceability.
2. Standardise column names.
3. Parse dates and ratings into appropriate types.
4. Remove exact duplicate rows only.
5. Preserve repeated IDs that are not exact duplicate rows for further investigation.
6. Exclude blank review text only from text-based sentiment and keyword analysis; rating analysis can still use rows with valid ratings.

In [ ]:
df = raw_df.rename(columns={
    "Review Id": "review_id",
    "Review": "review",
    "Ratings": "rating",
    "Review Date": "review_date",
}).copy()

df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["review"] = df["review"].fillna("").astype(str).str.strip()

rows_before_dedup = len(df)
df = df.drop_duplicates().copy()
exact_duplicates_removed = rows_before_dedup - len(df)

# Records for rating/date analysis
analysis_df = df.dropna(subset=["rating", "review_date"]).copy()

# Records with actual text for NLP
text_df = analysis_df.loc[analysis_df["review"].ne("")].copy()

print(f"Rows before exact-deduplication: {rows_before_dedup:,}")
print(f"Exact duplicate rows removed: {exact_duplicates_removed:,}")
print(f"Rows after exact-deduplication: {len(df):,}")
print(f"Rows available for rating/date analysis: {len(analysis_df):,}")
print(f"Non-empty reviews for NLP: {len(text_df):,}")
print("Repeated IDs still requiring review:", analysis_df["review_id"].duplicated().sum())

## 5. Rating analysis (EDA)

Calculate rating counts, percentages, and average rating. These are rating metrics, not sentiment labels.

In [ ]:
rating_counts = analysis_df["rating"].value_counts().sort_index()
rating_pct = analysis_df["rating"].value_counts(normalize=True).sort_index().mul(100)

rating_summary = pd.DataFrame({
    "review_count": rating_counts,
    "percentage": rating_pct.round(2),
})
display(rating_summary)
print(f"Average rating: {analysis_df['rating'].mean():.3f} / 5")

ax = rating_counts.plot(kind="bar", figsize=(8, 4), title="Review count by star rating")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. TextBlob sentiment analysis

TextBlob provides two scores:
- **Polarity:** from -1 (negative) to +1 (positive).
- **Subjectivity:** from 0 (more objective) to 1 (more subjective).

For a baseline classification consistent with the project brief, we use polarity `> 0` as Positive, `< 0` as Negative, and exactly `0` as Neutral. This is a simple rule and should be discussed as a limitation.

In [ ]:
def get_sentiment_scores(text):
    sentiment = TextBlob(text).sentiment
    return pd.Series({
        "polarity": sentiment.polarity,
        "subjectivity": sentiment.subjectivity,
    })

scores = text_df["review"].apply(get_sentiment_scores)
text_df = pd.concat([text_df, scores], axis=1)

text_df["sentiment"] = np.select(
    [text_df["polarity"] > 0, text_df["polarity"] < 0],
    ["Positive", "Negative"],
    default="Neutral",
)

print("Sentiment scores calculated for", f"{len(text_df):,}", "reviews")
display(text_df[["review", "rating", "polarity", "subjectivity", "sentiment"]].head())

## 7. Sentiment summary and visualisation

This cell generates the sentiment counts and shares used in the presentation. These outputs are calculated from the current analysis dataset.

In [ ]:
sentiment_order = ["Positive", "Neutral", "Negative"]
sentiment_counts = text_df["sentiment"].value_counts().reindex(sentiment_order, fill_value=0)
sentiment_pct = text_df["sentiment"].value_counts(normalize=True).reindex(sentiment_order, fill_value=0).mul(100)
sentiment_summary = pd.DataFrame({
    "review_count": sentiment_counts,
    "percentage": sentiment_pct.round(2),
})
display(sentiment_summary)
print(f"Average polarity: {text_df['polarity'].mean():.4f}")
print(f"Average subjectivity: {text_df['subjectivity'].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sentiment_counts.plot(kind="bar", ax=axes[0], color=["#2e8b57", "#9e9e9e", "#c94c4c"])
axes[0].set_title("Sentiment classification")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Review count")
axes[0].tick_params(axis="x", rotation=0)
axes[1].hist(text_df["polarity"], bins=40, edgecolor="white")
axes[1].set_title("Distribution of sentiment polarity")
axes[1].set_xlabel("Polarity")
axes[1].set_ylabel("Review count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(text_df["subjectivity"], bins=30, edgecolor="white")
plt.title("Distribution of subjectivity")
plt.xlabel("Subjectivity (0 = objective, 1 = subjective)")
plt.ylabel("Review count")
plt.tight_layout()
plt.show()

## 8. Compare star ratings with text sentiment

This cross-tab answers: Do star ratings and written sentiment always agree? Values below are row percentages, so each rating row totals approximately 100%.

In [ ]:
rating_sentiment_counts = pd.crosstab(text_df["rating"], text_df["sentiment"]).reindex(columns=sentiment_order, fill_value=0)
rating_sentiment_pct = pd.crosstab(text_df["rating"], text_df["sentiment"], normalize="index").reindex(columns=sentiment_order, fill_value=0).mul(100)

display(rating_sentiment_counts)
display(rating_sentiment_pct.round(2))

plt.figure(figsize=(8, 5))
sns.heatmap(rating_sentiment_pct, annot=True, fmt=".1f", cmap="Blues")
plt.title("Sentiment mix within each star rating (%)")
plt.xlabel("Text sentiment")
plt.ylabel("Star rating")
plt.tight_layout()
plt.show()

## 9. Keyword and phrase extraction

Use `CountVectorizer` to count common unigrams and bigrams separately in positive and negative reviews. We remove common English stop words and a small set of generic app/product terms. Review the resulting words in context before turning them into product recommendations.

In [ ]:
custom_stop_words = set(ENGLISH_STOP_WORDS).union({
    "app", "chatgpt", "chat", "gpt", "ai", "use", "using"
})

def get_top_terms(series, ngram_range=(1, 1), top_n=20, min_df=5):
    series = series.dropna().astype(str)
    if series.empty:
        return pd.DataFrame(columns=["term", "frequency"])
    vectorizer = CountVectorizer(
        stop_words=list(custom_stop_words),
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=10000,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )
    matrix = vectorizer.fit_transform(series)
    frequencies = np.asarray(matrix.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    order = np.argsort(frequencies)[::-1][:top_n]
    return pd.DataFrame({"term": terms[order], "frequency": frequencies[order]})

positive_reviews = text_df.loc[text_df["sentiment"].eq("Positive"), "review"]
negative_reviews = text_df.loc[text_df["sentiment"].eq("Negative"), "review"]

positive_terms = get_top_terms(positive_reviews, (1, 2), top_n=20)
negative_terms = get_top_terms(negative_reviews, (1, 2), top_n=20)

print("Most frequent positive-review terms")
display(positive_terms)
print("Most frequent negative-review terms")
display(negative_terms)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, data, title in [
    (axes[0], positive_terms.head(15), "Positive review terms"),
    (axes[1], negative_terms.head(15), "Negative review terms"),
]:
    data_plot = data.sort_values("frequency")
    ax.barh(data_plot["term"], data_plot["frequency"])
    ax.set_title(title)
    ax.set_xlabel("Frequency")
plt.tight_layout()
plt.show()

## 10. Monthly review and sentiment trends

Aggregate review volume, average polarity, average subjectivity, and average rating by calendar month.

In [ ]:
text_df["month"] = text_df["review_date"].dt.to_period("M").astype(str)
monthly_summary = (
    text_df.groupby("month")
    .agg(
        review_count=("review_id", "size"),
        average_polarity=("polarity", "mean"),
        average_subjectivity=("subjectivity", "mean"),
        average_rating=("rating", "mean"),
    )
    .reset_index()
)
display(monthly_summary.round(4))

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(monthly_summary["month"], monthly_summary["review_count"], marker="o")
axes[0].set_title("Monthly review volume")
axes[0].set_ylabel("Number of reviews")
axes[1].plot(monthly_summary["month"], monthly_summary["average_polarity"], marker="o", label="Average polarity")
axes[1].plot(monthly_summary["month"], monthly_summary["average_subjectivity"], marker="o", label="Average subjectivity")
axes[1].set_title("Monthly average sentiment scores")
axes[1].set_ylabel("Score")
axes[1].legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Automatically generate presentation findings

Run this after all previous cells. It prints the key values to verify before using them in your video script.

In [ ]:
top_rating = rating_pct.idxmax()
lowest_volume_month = monthly_summary.loc[monthly_summary["review_count"].idxmin()]
highest_volume_month = monthly_summary.loc[monthly_summary["review_count"].idxmax()]
lowest_polarity_month = monthly_summary.loc[monthly_summary["average_polarity"].idxmin()]
highest_polarity_month = monthly_summary.loc[monthly_summary["average_polarity"].idxmax()]

print("PRESENTATION FACTS — GENERATED FROM THIS DATASET")
print("=" * 55)
print(f"Raw records: {len(raw_df):,}")
print(f"Exact duplicate rows removed for analysis: {exact_duplicates_removed:,}")
print(f"Records after exact deduplication: {len(df):,}")
print(f"Text reviews analysed: {len(text_df):,}")
print(f"Average rating: {analysis_df['rating'].mean():.2f} / 5")
print(f"Most common rating: {top_rating} stars ({rating_pct.loc[top_rating]:.2f}%)")
print(f"Positive sentiment: {sentiment_pct['Positive']:.2f}% ({sentiment_counts['Positive']:,})")
print(f"Neutral sentiment: {sentiment_pct['Neutral']:.2f}% ({sentiment_counts['Neutral']:,})")
print(f"Negative sentiment: {sentiment_pct['Negative']:.2f}% ({sentiment_counts['Negative']:,})")
print(f"Average polarity: {text_df['polarity'].mean():.4f}")
print(f"Average subjectivity: {text_df['subjectivity'].mean():.4f}")
print(f"Highest review-volume month: {highest_volume_month['month']} ({highest_volume_month['review_count']:,})")
print(f"Lowest review-volume month: {lowest_volume_month['month']} ({lowest_volume_month['review_count']:,})")
print(f"Lowest mean-polarity month: {lowest_polarity_month['month']} ({lowest_polarity_month['average_polarity']:.4f})")
print(f"Highest mean-polarity month: {highest_polarity_month['month']} ({highest_polarity_month['average_polarity']:.4f})")
print("\nRating-to-sentiment percentage table:")
display(rating_sentiment_pct.round(2))

## 12. Interpretation and limitations

- A star rating and text sentiment measure different things; do not use them interchangeably.
- TextBlob is a baseline lexicon/rule-based tool and may misclassify sarcasm, context, negation, mixed opinions, emojis, and very short reviews.
- Word frequency shows which terms appear often, not whether a feature caused satisfaction or dissatisfaction. Inspect example reviews around important terms.
- Review volume varies over time; volume changes do not by themselves prove changes in user satisfaction.
- This dataset covers July 2023 to August 2024. Avoid claiming long-term or recurring seasonality.
- Repeated Review IDs that remain after exact-row deduplication should be investigated. Do not silently delete them without a defined rule.

## 13. Suggested presentation flow

1. Business problem and objectives.
2. Dataset and date coverage.
3. Data-quality audit and cleaning decisions.
4. Rating distribution.
5. Polarity, subjectivity, and sentiment categories.
6. Rating-versus-sentiment comparison.
7. Positive and negative terms.
8. Monthly review volume and polarity.
9. Business implications and limitations.

Use the printed facts from Section 11 for your final narration. Do not quote any result that has not been reproduced by running the notebook.